In [1]:
# Ячейка 1 - Импорт
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout, Input, MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score
import pickle

# Загружаем данные
X_train = np.load('../../data/X_train.npy')
X_test = np.load('../../data/X_test.npy')
y_train = np.load('../../data/y_train.npy')
y_test = np.load('../../data/y_test.npy')

with open('../../data/label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

NUM_CLASSES = len(le.classes_)
y_train_cat = to_categorical(y_train, NUM_CLASSES)

print(f"Данные загружены! Train: {X_train.shape}")

Данные загружены! Train: (638376, 10, 5)


In [2]:
# Ячейка 2 - Bidirectional LSTM + Attention
inputs = Input(shape=(10, 5))

# Bidirectional LSTM слои
x = Bidirectional(LSTM(128, return_sequences=True))(inputs)
x = Dropout(0.2)(x)
x = Bidirectional(LSTM(64, return_sequences=True))(x)
x = Dropout(0.2)(x)

# Attention механизм
attention = MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
x = LayerNormalization()(attention + x)

# Финальные слои
x = GlobalAveragePooling1D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model_v3 = Model(inputs, outputs)
model_v3.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_v3.summary()

2026-04-16 20:39:55.980916: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-04-16 20:39:55.981414: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-04-16 20:39:55.981422: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-04-16 20:39:55.981862: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-16 20:39:55.982441: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 10, 5)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 10, 256)   │    137,216 │ input_layer[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 10, 256)   │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 10, 128)   │    164,352 │ dropout[0][0]     │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 10, 128)   │          0 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 10, 128)   │     33,088 │ dropout_1[0][0],  │
│ (MultiHeadAttentio… │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 10, 128)   │          0 │ multi_head_atten… │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 10, 128)   │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     16,512 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      8,256 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 64)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 10)        │        650 │ dropout_4[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 360,330 (1.37 MB)

 Trainable params: 360,330 (1.37 MB)

 Non-trainable params: 0 (0.00 B)

In [3]:
# Ячейка 3 - Обучение
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1)
]

history = model_v3.fit(
    X_train, y_train_cat,
    epochs=30,
    batch_size=512,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/30


2026-04-16 20:40:29.190727: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1123/1123 ━━━━━━━━━━━━━━━━━━━━ 1295s 1s/step - accuracy: 0.5458 - loss: 1.5930 - val_accuracy: 0.6816 - val_loss: 1.0345 - learning_rate: 0.0010
Epoch 2/30
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 1293s 1s/step - accuracy: 0.6573 - loss: 1.0941 - val_accuracy: 0.6817 - val_loss: 0.9889 - learning_rate: 0.0010
Epoch 3/30
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 1322s 1s/step - accuracy: 0.6722 - loss: 1.0314 - val_accuracy: 0.6851 - val_loss: 1.0296 - learning_rate: 0.0010
Epoch 4/30
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 1294s 1s/step - accuracy: 0.6728 - loss: 1.0325 - val_accuracy: 0.6818 - val_loss: 0.9657 - learning_rate: 0.0010
Epoch 5/30
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 1286s 1s/step - accuracy: 0.6718 - loss: 1.0392 - val_accuracy: 0.6425 - val_loss: 1.0859 - learning_rate: 0.0010
Epoch 6/30
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 1300s 1s/step - accuracy: 0.6659 - loss: 1.0820 - val_accuracy: 0.6316 - val_loss: 1.2165 - learning_rate: 0.0010
Epoch 7/30
1123/1123 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6636 - l